In [ ]:
# 匯入相關套件
import os
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoAlertPresentException, TimeoutException


# region
def wait_and_click(xpath, timeout=5, use_js=False, use_click=True):
    """
    等待指定 xpath 的元素出現並點擊。

    Args:
        xpath (str): XPath 路徑
        timeout (int): 最多等待秒數
        use_js (bool): 是否用 JavaScript 點擊（預設 False）
    """
    if use_click:
        element = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.XPATH, xpath))
        )
    else:
        element = WebDriverWait(driver, timeout).until(
            EC.visibility_of_element_located((By.XPATH, xpath))
        )
    if use_js:
        driver.execute_script("arguments[0].click();", element)
    else:
        element.click()


def wait_and_select(xpath, value, timeout=5):
    """
    等待指定 xpath 的元素出現並選擇指定選項。

    Args:
        xpath (str): XPath 路徑
        value (str): 要選擇的選項值
        timeout (int): 最多等待秒數
    """
    element = WebDriverWait(driver, timeout).until(
        EC.visibility_of_element_located((By.XPATH, xpath))
    )
    Select(element).select_by_value(value)


def select_seats():
    """
    根據票價找到對應的座位區 group，
    再依 priority_seats 優先順序選擇區域。
    """

    # 依票價找到對應的 zone-label
    label = WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.XPATH, f'//div[contains(@class, "zone-label")][.//b[contains(normalize-space(), "{price}")]]'))
    )

    # 取得 group_id
    group_id = label.get_attribute("data-id")

    # 找到該票價的區域
    available_areas = driver.find_element(By.ID, group_id).find_elements(By.CSS_SELECTOR, "li.select_form_a a")

    available_names = {area.text.strip(): area for area in available_areas}

    # 按優先順序尋找
    for priority in priority_seats:
        matched = next(
            (area for name, area in available_names.items() if priority in name), None
        )

        if matched:
            driver.execute_script("arguments[0].click();", matched)

            return

    # 沒找到任何優先區域
    print("❌ 優先區域皆無票")
    input("請人工選擇區域後按 Enter 繼續...")


def fuzzy_match(
    outer_attr, outer_keyword, inner_attr=None, inner_keyword=None, click=True
):
    """
    找出內層符合 inner_attr ≈ inner_keyword 的元素，若 click=True 自動 scroll + 點擊。
    忽略空格、標點、大小寫差異。
    """
    global driver
    script = """
        const norm = s => s ? s.replace(/\\W+/g, "").toLowerCase() : "";
        const outerKey = norm(arguments[0]);
        const innerKey = norm(arguments[1]);
        const outerAttr = arguments[2];
        const innerAttr = arguments[3];
        const shouldClick = arguments[4];

        const outer = Array.from(document.querySelectorAll(`[${outerAttr}]`))
            .find(el => norm(el.getAttribute(outerAttr)).includes(outerKey));
        
        if (!outer) return null;

        if (!innerAttr) {
            if (shouldClick) {
                outer.scrollIntoView({block: "center"});
                setTimeout(() => outer.click(), 100);
            }
            return shouldClick ? null : outer;
        }

        const matched = Array.from(outer.querySelectorAll(`[${innerAttr}]`))
            .find(el => norm(el.getAttribute(innerAttr)).includes(innerKey));
        
        if (matched && shouldClick) {
            matched.scrollIntoView({block: "center"});
            setTimeout(() => matched.click(), 100);
        }

        return shouldClick ? null : matched;
    """

    return driver.execute_script(
        script, outer_keyword, inner_keyword or "", outer_attr, inner_attr or "", click
    )


# endregion

In [ ]:
# 初始設定
load_dotenv()
TARGET_TIME = "2026-09-19 13:00:00"
target_name = "26_aaa"
URL = f"https://tixcraft.com/activity/detail/{target_name}"
EMAIL = os.getenv("EMAIL")
PASSWORD = os.getenv("PASSWORD")
CARD_NUMBER = os.getenv("CARD_NUMBER")
CARD_EXPIRY = os.getenv("CARD_EXPIRY")
MAX_RETRIES = 100
priority_seats = [
    "C02", "C10",
    "E01", "E02", "E14", "E15",
    "G02", "G12",
    "V10", "V03"
]
price = 5980

In [ ]:
# test
# TARGET_TIME = "2026-09-18 13:00:00"
# target_name = "26_patrick"
# URL = f"https://tixcraft.com/activity/detail/{target_name}"
# priority_seats = [
#     "3A"
# ]
# group_id = "group_3"

In [ ]:
# 預先準備
options = Options()
driver = webdriver.Chrome(options=options)

# 檢查狀態
driver.get(URL)
WebDriverWait(driver, 3).until(
    lambda d: d.execute_script("return document.readyState") == "complete"
)

# 登入流程
wait_and_click('//*[@href="#login"]')
print("🔐 點擊登入按鈕")
wait_and_click('//*[@id="loginFacebook"]')
print("🔗 點擊 FB 登入")

# 輸入帳密並登入
print("📝 輸入帳號密碼登入...")
WebDriverWait(driver, 5).until(
    EC.visibility_of_element_located((By.XPATH, '//*[@name="email"]'))
)
driver.find_element(By.XPATH, '//*[@name="email"]').send_keys(EMAIL)
driver.find_element(By.XPATH, '//*[@name="pass"]').send_keys(PASSWORD)
wait_and_click('//*[text()="登入"]')
print("✅ 登入")
input("🔐 人工驗證")
wait_and_click('//*[@aria-label="以豆的身分繼續"]')
print("➡️ 點擊以豆的身分繼續")

In [ ]:
# 等待指定時間
driver.get(URL)
target_dt = datetime.fromisoformat(TARGET_TIME)
refresh_start_dt = target_dt - timedelta(minutes=1)
print(f"⏳ 等待指定時間...")
while datetime.now() < refresh_start_dt:
    time.sleep(0.1)
print("🔄 開始刷新...")

In [ ]:
# 持續刷新
while True:
    driver.refresh()

    try:
        wait_and_click(f'//*[@href="/activity/game/{target_name}"]', 1)

        WebDriverWait(driver, 0.5).until(
            EC.element_to_be_clickable(
                (
                    By.XPATH,
                    f'//*[starts-with(@data-href, "https://tixcraft.com/ticket/area/{target_name}")]',
                )
            )
        )
        break

    except TimeoutException:
        print("⏳ 倒數中，重新刷新...")
        time.sleep(0.2)

In [ ]:
# 購票流程 - 選擇區域
print("🎫 立即購票")

wait_and_click(f'//*[starts-with(@data-href, "https://tixcraft.com/ticket/area/{target_name}")]', use_js = True)
print("🛒 立即訂購")

select_seats()
print("📍 選擇區域")

In [ ]:
# 購票流程 - 選擇票數
while True:
    wait_and_select('//*[starts-with(@id, "TicketForm_ticketPrice_")]', '1')
    print("🎟️ 選擇票數")

    driver.find_element(By.XPATH,'//*[@id="TicketForm_verifyCode"]').send_keys(input("🔐 輸入驗證碼："))

    wait_and_click('//*[@id="TicketForm_agree"]')
    print("☑️ 同意服務條款")
        
    wait_and_click('//*[text()="確認張數"]', use_js = True)
    print("✅ 確認張數")

    time.sleep(1)
    try:
        alert = driver.switch_to.alert
        print(f"⚠️ {alert.text}")
        alert.accept()
        print("🔄 重新嘗試...")
        
    except NoAlertPresentException:
        print("🎉 沒有錯誤視窗，進入下一步")
        break

wait_and_click('//*[@id="CheckoutForm_paymentId_36"]', 60)
print("💳 選擇付款方式")

time.sleep(1)

wait_and_click('//*[@id="submitButton"]', use_js=True)
print("✅ 下一步")

In [ ]:
# 結帳
driver.find_element(By.XPATH, '//*[@id="cardNumber"]').send_keys(CARD_NUMBER)
print("💳 信用卡卡號")

wait_and_select('//*[@id="ExpirationMonth"]', CARD_EXPIRY.split('/')[0])
wait_and_select('//*[@id="ExpirationYear"]', "20" + CARD_EXPIRY.split('/')[1])
print("💳 卡片到期日")

driver.find_element(By.XPATH, '//*[@id="check_num"]').send_keys(input("💳 輸入卡片檢查碼"))
print("💳 卡片檢查碼")

wait_and_click('//*[@id="btn_box"]')
print("✅ 結帳")